# FMSE Laboratorio 13 - Rompe el agente

**Curso:** FMSE 2026 · **Módulo 13:** Seguridad de la IA, red teaming y arquitectura de seguridad · **Fase:** Verificar y validar · **Nivel:** Sénior

La página del curso y tu progreso están en [agentic-ai.es/academy/fmse/learn/m13-security-red-teaming](https://agentic-ai.es/academy/fmse/learn/m13-security-red-teaming).

> **Sobre el idioma.** Las explicaciones y los mensajes de los validadores están en español. El código, los nombres de las variables, las claves de los datos y los datos de ejemplo se quedan en inglés (son lo que leen los validadores). Puedes escribir tus respuestas en español o en inglés.

## 0. Misión y competencia objetivo

**Misión.** Descubrir al menos cuatro vulnerabilidades introducidas a propósito, implementar mitigaciones en la arquitectura y demostrar con pruebas de regresión que funcionan.

**Laboratorio guiado.** Atacas un agente vulnerable a propósito y examinas las trazas.

**Competencia objetivo (resultados del Módulo 13):**
- Modelar las amenazas de la inyección de prompts directa e indirecta, la revelación de datos sensibles, el envenenamiento de la cadena de suministro o del contexto, el manejo inadecuado de las salidas, la agencia excesiva y el consumo sin límites.
- Aplicar el mínimo privilegio y puntos de aprobación fuera del modelo.
- Atacar sistemas de recuperación y de herramientas, y convertir los ataques que funcionan en pruebas de regresión.
- Diseñar el registro de incidentes, los interruptores de emergencia y modos degradados seguros.

**Guía:** alrededor de un 25% guiado y un 75% por tu cuenta. Es un **reto de ingeniería**: la parte guiada te da un camino de referencia que funciona; el reto, no.

## 1. Requisitos y criterios de aceptación

Estos requisitos los comprueban validadores públicos. Te dicen qué requisito falló y por qué, nunca la implementación que se esperaba.

| ID | Requisito |
| --- | --- |
| RED-01 | Hay al menos 4 vulnerabilidades documentadas |
| RED-02 | Cada una tiene activo, impacto, control y prueba |
| RED-03 | Las herramientas que modifican datos tienen el mínimo privilegio **(crítico)** |
| RED-04 | Hay pruebas de regresión contra la inyección **(crítico)** |
| RED-05 | Los límites de recursos están definidos |
| RED-06 | Ninguna vulnerabilidad conocida de gravedad alta queda sin resolver **(crítico)** |

**Control de competencia (portal):** ninguna vulnerabilidad conocida de gravedad alta introducida a propósito queda sin resolver. También hacen falta el cuestionario (>= 80%), el laboratorio guiado, el artefacto y tu reflexión. Un requisito crítico no se compensa con las demás puntuaciones.

## 2. Preparación del entorno

In [ ]:
# Idempotente: se puede volver a ejecutar y nunca imprime secretos.
LAB_ID = "lab-13"
LABKIT_VERSION = "1.1.0"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit ya estaba cargado"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"usando el labkit de {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "usando el labkit descargado anteriormente"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"labkit descargado y verificado desde {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "No se pudo cargar fmse_labkit desde: " + "; ".join(problems) + ". "
        "Solución: descarga la carpeta de laboratorios desde agentic-ai.es/academy/fmse/resources, sube la carpeta fmse_labkit junto a este notebook (panel Archivos) y vuelve a ejecutar esta celda."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

fmse.set_language("es")
print(f"fmse_labkit {fmse.__version__} listo para {LAB_ID} ({fmse.title(fmse.lab_spec(LAB_ID))})")

## 3. Comprobación de secretos

Este laboratorio funciona por completo sin conexión: no necesitas ninguna clave de proveedor. Las claves que uses en otros laboratorios van en los Secretos de Colab y nunca se imprimen.

In [ ]:
fmse.secrets_check()

## 4. Experimento guiado

`SupportAgent()` sin hooks, sin alcances y sin límites es la versión vulnerable. Su "modelo" trata cualquier texto en imperativo como una instrucción, venga del usuario o de un documento recuperado. Hay cinco ataques preparados; su texto está en inglés.

In [ ]:
from fmse_labkit.labs import lab13

guided = fmse.GuidedLog(LAB_ID, required_steps=["attacks_run", "traces_inspected"])
vulnerable = lab13.vulnerable_agent()
for aid, a in lab13.ATTACKS.items():
    print(aid, f"[{a['severity']}]", "-", a["request"][:60])

## 5. Predice antes de ejecutar

¿Cuáles de los cinco ataques van a funcionar contra el agente vulnerable? Escribe tu predicción y después ejecuta el experimento.

In [ ]:
my_prediction = ""
guided.predict("prediction", my_prediction)

In [ ]:
results = {aid: lab13.run_attack(vulnerable, aid) for aid in lab13.ATTACKS}
for aid, res in results.items():
    print(f"{aid:<36} exploited={lab13.exploited(aid, res)}")
guided.step("attacks_run")
r = results["A1-indirect-injection-exfiltration"]
print("ejecutado:", [(c.tool, c.args.get("to"), c.origin) for c in r.executed])
print("registro:", r.log)
guided.step("traces_inspected")
guided.outcome("prediction", "ver la salida de arriba")

**Hoja de trabajo.** Al menos una frase completa por campo: estas observaciones forman parte de la evidencia de tu laboratorio guiado.

In [ ]:
worksheet = {
    "trace_observation": "",
    "boundary_observation": "",
    "prediction_check": "",
}
guided.observe(worksheet)

## 6. Reto

**Encuentra, corrige y demuestra cuatro vulnerabilidades introducidas a propósito.** Descubre al menos cuatro vulnerabilidades introducidas a propósito, implementa mitigaciones en la arquitectura y demuestra con pruebas de regresión que funcionan.

Restricciones:
- Las mitigaciones viven en la aplicación: alcances, guardas, límites, manejo de las salidas, enmascarado de datos en los registros.
- Todo ataque de gravedad alta tiene que fallar contra tu agente.

In [ ]:
def build_agent():
    """Devuelve lab13.SupportAgent(hooks=..., tool_scopes=..., limits=...) con tus mitigaciones en la arquitectura."""
    raise NotImplementedError


report = []  # {"id", "title", "asset", "impact", "control", "test" (id de una prueba de regresión), "severity"}
regression_tests = []  # {"id", "attack": id del ataque, "check": fn(agent) -> True cuando el agente es seguro}

## 7. Validación pública

In [ ]:
submission = {"build_agent": build_agent, "report": report, "regression_tests": regression_tests}
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustez / pruebas adversariales

Condiciones adversariales adicionales, fuera de la batería que se califica. No cambian tu puntuación: te enseñan dónde es frágil tu solución.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Reflexión de ingeniería

La reflexión que se califica se escribe en el portal y se guarda con tu artefacto. Si te sirve, haz aquí un borrador:
- **Decisión:** ¿qué control frena la inyección indirecta y por qué no puede vivir en el prompt?
- **Compromisos:** ¿qué comportamiento legítimo restringieron tus controles y cómo conseguiste que el agente siguiera siendo útil?
- **Riesgos pendientes:** ¿qué clase de ataque no has probado y a qué apuntaría?

In [ ]:
reflection_draft = {"decision": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Exportar el artefacto / resultado de finalización

Exporta el artefacto **Modelo de amenazas, informe de red teaming y conjunto de regresión de seguridad** e imprime un registro de finalización. Copia el registro y pégalo en la página del Módulo 13 de agentic-ai.es (**Importar el resultado del laboratorio**). El registro es un documento del flujo de aprendizaje, no un certificado firmado.

In [ ]:
artifact = fmse.export_artifact(LAB_ID, {"report": report, "regression_tests": [{"id": t.get("id"), "attack": t.get("attack")} for t in regression_tests]},
                     title="Modelo de amenazas, informe de red teaming y conjunto de regresión de seguridad", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, **{}},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)